# Phase 2 Code Challenge

This code challenge is designed to test your understanding of the Phase 2 material. It covers:

- SQL
- Bayesian Statistics
- Normal Distribution
- Statistical Tests

_Read the instructions carefully_. You will be asked both to write code and to answer short answer questions.

## Code Tests

We have provided some code tests for you to run to check that your work meets the item specifications. Passing these tests does not necessarily mean that you have gotten the item correct - there are additional hidden tests. However, if any of the tests do not pass, this tells you that your code is incorrect and needs changes to meet the specification. To determine what the issue is, read the comments in the code test cells, the error message you receive, and the item instructions.

## Short Answer Questions 

For the short answer questions...

* _Use your own words_. It is OK to refer to outside resources when crafting your response, but _do not copy text from another source_.

* _Communicate clearly_. We are not grading your writing skills, but you can only receive full credit if your teacher is able to fully understand your response. 

* _Be concise_. You should be able to answer most short answer questions in a sentence or two. Writing unnecessarily long answers increases the risk of you being unclear or saying something incorrect.

In [58]:
# Run this cell without changes to import the necessary libraries

import itertools
import numpy as np
import pandas as pd 
from numbers import Number
import sqlite3
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import pickle

---
## Part 1: SQL [Suggested time: 20 minutes]
---
In this part, you will create and execute three SQL queries on the Chinook database. For this challenge **you will need to access the `Album` and `Artist` tables**.

### 1.1) Connect to the Database.

In [59]:
# CodeGrade step1.1
# Replace None with appropriate code
# Connect to the Database here ("Chinook_Sqlite.sqlite")

path = "Chinook_Sqlite.sqlite"
conn = sqlite3.connect(path)

In [60]:
assert type(path) == str

In [61]:
# Run this cell without changes to see all the
# tables in the database.

df = pd.read_sql(
    """
    SELECT *
    FROM sqlite_master
    """
, conn
)

df[df['type'] == 'table']

,type,name,tbl_name,rootpage,sql
0,table,Album,Album,2,CREATE TABLE [Album]\n(\n [AlbumId] INTEGER...
1,table,Artist,Artist,3,CREATE TABLE [Artist]\n(\n [ArtistId] INTEG...
2,table,Customer,Customer,4,CREATE TABLE [Customer]\n(\n [CustomerId] I...
3,table,Employee,Employee,7,CREATE TABLE [Employee]\n(\n [EmployeeId] I...
4,table,Genre,Genre,9,CREATE TABLE [Genre]\n(\n [GenreId] INTEGER...
5,table,Invoice,Invoice,10,CREATE TABLE [Invoice]\n(\n [InvoiceId] INT...
6,table,InvoiceLine,InvoiceLine,12,CREATE TABLE [InvoiceLine]\n(\n [InvoiceLin...
7,table,MediaType,MediaType,14,CREATE TABLE [MediaType]\n(\n [MediaTypeId]...
8,table,Playlist,Playlist,15,CREATE TABLE [Playlist]\n(\n [PlaylistId] I...
9,table,PlaylistTrack,PlaylistTrack,16,CREATE TABLE [PlaylistTrack]\n(\n [Playlist...


### 1.2) Write a query to return the last ten artists alphabetically.

In [62]:

first_query = """SELECT *
                 FROM Artist;"""

# Use the line below to check your query's output
pd.read_sql(first_query, conn).head()

,ArtistId,Name
0,1,AC/DC
1,2,Accept
2,3,Aerosmith
3,4,Alanis Morissette
4,5,Alice In Chains


In [63]:
# CodeGrade step1.2
# Replace None with appropriate code
# Hint: Use the Artist table!

first_query = """SELECT *
                  FROM Artist
                  ORDER BY Name ASC;"""
                  
pd.read_sql(first_query, conn).tail(10)

,ArtistId,Name
265,71,Vinícius De Moraes & Baden Powell
266,74,Vinícius E Odette Lara
267,73,Vinícius E Qurteto Em Cy
268,154,Whitesnake
269,211,Wilhelm Kempff
270,181,Xis
271,255,Yehudi Menuhin
272,212,Yo-Yo Ma
273,168,Youssou N'Dour
274,155,Zeca Pagodinho


In [64]:
# first_query should be a string
assert type(first_query) == str

# first_query should be a SQL query
first_query_df = pd.read_sql(first_query, conn)

### 1.3) Write a query to return all the albums in the database from Led Zeppelin.

In [65]:
second_query = """SELECT *
                      FROM Album
                      ;"""

# Use the line below to check your query's output
pd.read_sql(second_query, conn).head()

,AlbumId,Title,ArtistId
0,1,For Those About To Rock We Salute You,1
1,2,Balls to the Wall,2
2,3,Restless and Wild,2
3,4,Let There Be Rock,1
4,5,Big Ones,3


In [66]:
# CodeGrade step1.3
# Replace None with appropriate code
# Hint: Use the Artist and Album tables!

second_query = """SELECT *
                  FROM Album
                  JOIN Artist USING (ArtistId)
                  WHERE Name = 'Led Zeppelin';"""

pd.read_sql(second_query, conn)

,AlbumId,Title,ArtistId,Name
0,30,BBC Sessions [Disc 1] [Live],22,Led Zeppelin
1,44,Physical Graffiti [Disc 1],22,Led Zeppelin
2,127,BBC Sessions [Disc 2] [Live],22,Led Zeppelin
3,128,Coda,22,Led Zeppelin
4,129,Houses Of The Holy,22,Led Zeppelin
5,130,In Through The Out Door,22,Led Zeppelin
6,131,IV,22,Led Zeppelin
7,132,Led Zeppelin I,22,Led Zeppelin
8,133,Led Zeppelin II,22,Led Zeppelin
9,134,Led Zeppelin III,22,Led Zeppelin


In [67]:
# second_query should be a string
assert type(second_query) == str

# second_query should be a SQL query
second_query_df = pd.read_sql(second_query, conn)

### 1.4) Write a query to return both the artist with the most albums in the database and the number of albums.

In [68]:
# CodeGrade step1.4
# Replace None with appropriate code

third_query = """SELECT Name ,Title, COUNT(*) AS number_albums
                  FROM Artist
                  JOIN Album USING (ArtistId);"""

pd.read_sql(third_query, conn)

,Name,Title,number_albums
0,AC/DC,For Those About To Rock We Salute You,347


In [69]:
# third_query should be a string
assert type(third_query) == str

# third_query should be a SQL query
third_query_df = pd.read_sql(third_query, conn)

---
## Part 2: Bayesian Statistics [Suggested time: 15 minutes]
---

A medical test is designed to diagnose a certain disease. The test has a false positive rate of 10%, meaning that 10% of people without the disease will get a positive test result. The test has a false negative rate of 2%, meaning that 2% of people with the disease will get a negative result. Only 1% of the population has this disease.

### 2.1) Create a numeric variable `p_pos_test` containing the probability of a person receiving a positive test result.

Assume that the person being tested is randomly selected from the broader population.

In [70]:
# CodeGrade step2.1
# Replace None with appropriate code

false_pos_rate = 0.1  # False positive rate
false_neg_rate = 0.02  # False negative rate
population_rate = 0.01  # Percentage of population with the disease

# Probabilities
P_disease = population_rate  # Probability of having the disease
P_no_disease = 1 - population_rate  # Probability of not having the disease
P_pos_given_disease = 1 - false_neg_rate  # Probability of positive test given disease
P_pos_given_no_disease = false_pos_rate  # Probability of positive test given no disease

# Calculating the probability of a positive test result
p_pos_test = (P_pos_given_disease * P_disease) + (P_pos_given_no_disease * P_no_disease)

p_pos_test


0.10880000000000001

In [71]:
# This test confirms that you have created a numeric variable named p_pos_test

assert isinstance(p_pos_test, Number)

In [72]:
# These tests confirm that p_pos_test is a value between 0 and 1

assert p_pos_test >= 0
assert p_pos_test <= 1

### 2.2) Create a numeric variable `p_disease_given_pos` containing the probability of a person actually having the disease if they receive a positive test result.

Assume that the person being tested is randomly selected from the broader population.

Hint: Use your answer to the previous question to help answer this one.

In [73]:
# CodeGrade step2.2
# Replace None with appropriate code
    

false_pos_rate = 0.1  # False positive rate
false_neg_rate = 0.02  # False negative rate
population_rate = 0.01  # Percentage of population with the disease

# Probabilities
P_disease = population_rate  # Probability of having the disease
P_no_disease = 1 - population_rate  # Probability of not having the disease
P_pos_given_disease = 1 - false_neg_rate  # Probability of positive test given disease
P_pos_given_no_disease = false_pos_rate  # Probability of positive test given no disease

# p_pos_test from previous step
p_pos_test = (P_pos_given_disease * P_disease) + (P_pos_given_no_disease * P_no_disease)

# Using Bayes' Theorem to calculate P(Disease | Positive Test)
p_disease_given_pos = (P_pos_given_disease * P_disease) / p_pos_test

p_disease_given_pos


0.09007352941176469

In [74]:
# This test confirms that you have created a numeric variable named p_disease_given_pos

assert isinstance(p_disease_given_pos, Number)

In [75]:
# These tests confirm that p_disease_given_pos is a value between 0 and 1

assert p_disease_given_pos >= 0
assert p_disease_given_pos <= 1

---
## Part 3: Normal Distribution [Suggested time: 20 minutes]
---
In this part, you will analyze check totals at a TexMex restaurant. We know that the population distribution of check totals for the TexMex restaurant is normally distributed with a mean of \\$20 and a standard deviation of \\$3. 

### 3.1) Create a numeric variable `z_score_26` containing the z-score for a \\$26 check. 

In [76]:
# CodeGrade step3.1
# Replace None with appropriate code

x = 26  # Check total
mu = 20  # Mean of check totals
sigma = 3  # Standard deviation

# Calculate z-score
z_score_26 = (x - mu) / sigma
z_score_26

2.0

In [77]:
z_score_26 = (26-20) / 3
z_score_26

2.0

In [78]:
# This test confirms that you have created a numeric variable named z_score_26

assert isinstance(z_score_26, Number)

### 3.2) Create a numeric variable `p_under_26` containing the approximate proportion of all checks that are less than \\$26.

Hint: Use the answer from the previous question along with the empirical rule, a Python function, or this [z-table](https://www.math.arizona.edu/~rsims/ma464/standardnormaltable.pdf).

In [79]:
# CodeGrade step3.2
# Replace None with appropriate code
from scipy.stats import norm
p_under_26 = norm.cdf(z_score_26)
p_under_26

0.9772498680518208

In [80]:
# This test confirms that you have created a numeric variable named p_under_26

assert isinstance(p_under_26, Number)

# These tests confirm that p_under_26 is a value between 0 and 1

assert p_under_26 >= 0
assert p_under_26 <= 1

### 3.3) Create numeric variables `conf_low` and `conf_high` containing the lower and upper bounds (respectively) of a 95% confidence interval for the mean of one waiter's check amounts using the information below. 

One week, a waiter gets 100 checks with a mean of \\$19 and a standard deviation of \\$3.

In [81]:
# CodeGrade step3.3
# Replace None with appropriate code

n = 100
mean = 19
std = 3

z = 1.96  # z-value for 95% confidence

# Calculate the standard error
SE = std / np.sqrt(n)

# Calculate the margin of error
ME = z * SE

# Calculate confidence interval bounds
conf_low = mean - ME
conf_high = mean + ME

print([[conf_low],[conf_high]])

[[18.412], [19.588]]


In [82]:
# These tests confirm that you have created numeric variables named conf_low and conf_high

assert isinstance(conf_low, Number)
assert isinstance(conf_high, Number)

# This test confirms that conf_low is below conf_high

assert conf_low < conf_high

# These statements print your answers for reference to help answer the next question

print('The lower bound of the 95% confidence interval is {}'.format(conf_low))
print('The upper bound of the 95% confidence interval is {}'.format(conf_high))

The lower bound of the 95% confidence interval is 18.412
The upper bound of the 95% confidence interval is 19.588


### 3.4) Short Answer: Interpret the 95% confidence interval you just calculated in Question 1.3.

# Your answer here



---
## Part 4: Statistical Testing [Suggested time: 20 minutes]
---
The TexMex restaurant recently introduced queso to its menu.

We have a random sample containing 2000 check totals, all from different customers: 1000 check totals for orders without queso ("no queso") and 1000 check totals for orders with queso ("queso").

In the cell below, we load the sample data for you into the arrays `no_queso` and `queso` for the "no queso" and "queso" order check totals, respectively.

In [83]:
# Run this cell without changes

# Load the sample data 
no_queso = pickle.load(open('./no_queso.pkl', 'rb'))
queso = pickle.load(open('./queso.pkl', 'rb'))

### 4.1) Short Answer: State null and alternative hypotheses to use for testing whether customers who order queso spend different amounts of money from customers who do not order queso.

```css
Null Hypothesis: The different customers can order queso

Alternative Hypothesis:  The checks for either with queso or without queso total to 2000, 1000 for those with queso and 1000 those without queso

```

### 4.2) Short Answer: What would it mean to make a Type I error for this specific hypothesis test?

Your answer should be _specific to this context,_  not a general statement of what Type I error is.

Type I error would mean concluding that the addition of queso has a statistically significant impact on the check total when, in reality, it does not. Essentially, we would be incorrectly rejecting the null hypothesis and assuming that queso affects the check total, leading us to think that adding queso changes customer spending pwhen this change may actually be due to random chance.

### 4.3) Create a numeric variable `p_value` containing the p-value associated with a statistical test of your hypotheses. 

You must identify and implement the correct statistical test for this scenario. You can assume the two samples have equal variances.

Hint: Use `scipy.stats` to calculate the answer - it has already been imported as `stats`. Relevant documentation can be found [here](https://docs.scipy.org/doc/scipy/reference/stats.html#statistical-tests).

In [84]:
# CodeGrade step4.3
# Replace None with appropriate code

_, p_value = stats.ttest_ind(no_queso, queso, equal_var=True)
p_value

1.29670967092511e-307

In [85]:
# This test confirms that you have created a numeric variable named p_value

assert isinstance(p_value, Number)

### 4.4) Short Answer: Can you reject the null hypothesis using a significance level of $\alpha$ = 0.05? Explain why or why not.

To determine whether you can reject the null hypothesis at a significance level of 
𝛼 = 0.05, you should compare the calculated p-value with α. 

If  p_value > α: You fail to reject the null hypothesis, indicating that there is not enough evidence to suggest a significant difference.

If  p_value > 0.05, you cannot reject the null hypothesis because the result is not statistically significant.